# 📊 Deep Research — Single Report Evaluation for the domain - `NLP`

This notebook evaluates one Deep Research `.md` report using **YESciEval rubrics**.

### Flow
1. Read the **question from the NLP CSV** 
2. Load the `.md` report as plain text → passed as `answer` to YESciEval
3. Run the YESciEval judge across all rubrics
4. Aggregate rubric scores → **category scores (0–1)** 
5. Display a score table and a **6-panel quality plot**
6. Save JSON + CSV outputs

---
**Prerequisites:** `pip install yescieval python-dotenv matplotlib numpy`

## 1 — Imports & Setup

In [ ]:
!pip install yescieval python-dotenv matplotlib -q

In [ ]:
import re, json, csv
from pathlib import Path
from typing import Dict, Tuple, List
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv
load_dotenv()
print("✅ Core imports loaded")

In [ ]:
from yescieval import CustomAutoJudge, VocabularyInjector, ExampleInjector
from yescieval.rubric.pointwise.depth      import TemporalPrecision, CausalReasoning, MechanisticUnderstanding
from yescieval.rubric.pointwise.breadth    import ContextCoverage, ScopeCoverage, DimensionCoverage, ScaleCoverage, MethodCoverage
from yescieval.rubric.pointwise.rigor      import EpistemicCalibration, ExplicitUncertainty, QuantitativeEvidenceAndUncertainty
from yescieval.rubric.pointwise.innovation import StateOfTheArtAndNovelty
from yescieval.rubric.pointwise.gap        import GapIdentification
print("✅ YESciEval imports loaded")

## 2 — Configuration

Set the report path, CSV path, and judge model.

In [ ]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────

REPORT_PATH   = "your_report_path_here.md" 
QUESTIONS_CSV = "your_questions_csv_path_here.csv" 
OUTPUT_DIR  = "your_output_dir_here" 
DOMAIN         = "nlp" # nlp or ecology
MODEL_ID       = "your_model_id_here"
DEVICE         = "cpu"              # "cpu" | "cuda" | "mps"
HF_TOKEN       = "your_hf_token_here"                   # only for gated models
MAX_NEW_TOKENS = 2048

# ──────────────────────────────────────────────────────────────────────────

# Category → list of (RubricClass, weight) tuples.
CATEGORIES: Dict[str, List[Tuple[type, float]]] = {
    "depth": [
        (TemporalPrecision,         1/3),
        (CausalReasoning,           1/3),
        (MechanisticUnderstanding,  1/3),
    ],
    "breadth": [
        (ContextCoverage,   1/5),
        (ScopeCoverage,     1/5),
        (DimensionCoverage, 1/5),
        (MethodCoverage, 1/5),
        (ScaleCoverage, 1/5)
    ],
    "rigor": [
        (EpistemicCalibration, 1/3),
        (ExplicitUncertainty, 1/3),
        (QuantitativeEvidenceAndUncertainty, 1/3)
    ],
    "innovation": [
        (StateOfTheArtAndNovelty, 1.0)
    ],
    "gap": [
        (GapIdentification, 1.0),
    ],
}

RATING_MIN, RATING_MAX = 1, 5

print(f"📄 Report : {REPORT_PATH}")
print(f"📋 CSV    : {QUESTIONS_CSV}")
print(f"🤖 Model  : {MODEL_ID}")
print(f"📐 Categories: {list(CATEGORIES.keys())}")

## 3 — Helper Functions

In [ ]:
def parse_config(stem: str) -> str:
    """Extracts dX_bY from filename e.g. 1_o3-mini_orkg_d1_b1 → d1_b1."""
    m = re.search(r'd(\d+)_b(\d+)', stem)
    return f"d{m.group(1)}_b{m.group(2)}" if m else 'd1_b1'


def parse_report_number(stem: str) -> int:
    """Extracts the leading number from filename e.g. 1_o3-mini → 1."""
    m = re.match(r'(\d+)[_\-]', stem)
    return int(m.group(1)) if m else -1


def load_question_from_csv(csv_path: str, report_number: int) -> Tuple[str, str]:
    """
    Reads (title/question, problem_statement) for a given report number from the NLP CSV.
    report_number=1 → first data row, report_number=2 → second data row, etc.

    Returns:
        title            – paper/report title string
    """
    encodings_to_try = ['cp1252', 'utf-8', 'utf-8-sig', 'latin-1']

    for encoding in encodings_to_try:
        try:
            with open(csv_path, newline='', encoding=encoding) as f:
                reader = csv.DictReader(f)
                for row_num, row in enumerate(reader, start=1):
                    if row_num == report_number:
                        normalised = {k.strip().lower(): v.strip() for k, v in row.items()}
                        title= normalised.get('title', '')
                        return title 
            return '', f'Row {report_number} not found in CSV'
        except UnicodeDecodeError:
            continue

    return '', 'Could not open CSV with any known encoding'


def parse_judge_result(result, rubric_name: str) -> Tuple[int, str]:
    """
    Extracts (rating 1-5, rationale) from a YESciEval judge result.
    Handles: raw string with optional <think> block, dict, Pydantic model.
    """
    if isinstance(result, str):
        cleaned = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
        json_str = None
        depth, start = 0, None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if depth == 0: start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start is not None:
                    json_str = cleaned[start:i+1]
                    break
        if json_str:
            try:
                result = json.loads(json_str)
            except json.JSONDecodeError:
                pass
        if isinstance(result, str):
            try: return int(result.strip()), ''
            except ValueError: return 0, result
    if isinstance(result, dict):
        inner = result.get(rubric_name) or result.get(rubric_name.lower())
        if inner is None and len(result) == 1:
            inner = next(iter(result.values()))
        if isinstance(inner, dict):
            return int(inner.get('rating', 0)), str(inner.get('rationale', ''))
        if 'rating' in result:
            return int(result['rating']), str(result.get('rationale', ''))
        return 0, str(result)
    if hasattr(result, 'rating') and hasattr(result, 'rationale'):
        return int(result.rating), str(result.rationale)
    if hasattr(result, 'score'):
        try: return int(round(float(result.score))), ''
        except (ValueError, TypeError): pass
    try: return int(str(result).strip()), ''
    except ValueError: pass
    return 0, str(result)


def compute_category_score(rubric_scores: List[Tuple[float, float]]) -> float:
    """
    Weighted mean of (score, weight) pairs, normalised from 1-5 to 0.0-1.0.
    Formula: (weighted_mean - 1) / (5 - 1)
    """
    total_w = sum(w for _, w in rubric_scores)
    if not total_w:
        return 0.0
    weighted_mean = sum(s * w for s, w in rubric_scores) / total_w
    return round((weighted_mean - 1) / (RATING_MAX - RATING_MIN), 4)


print("✅ Helpers defined")

## 4 — Load Report & Read Research Question from CSV

The **title** from the NLP CSV is used as the `question` input to YESciEval.

In [ ]:
report_path = Path(REPORT_PATH).resolve()
if not report_path.exists():
    raise FileNotFoundError(f'Report not found: {report_path}')

report_md  = report_path.read_text(encoding='utf-8', errors='ignore')
stem       = report_path.stem
config_str = parse_config(stem)
report_num = parse_report_number(stem)

question = load_question_from_csv(QUESTIONS_CSV, report_num)

print(f"📄 Report         : {report_path.name}")
print(f"📐 Config         : {config_str}")
print(f"🔢 Number         : {report_num}")
print(f"❓ Question (title): {question}")

## 5 — Initialise the YESciEval Judge

In [ ]:
print(f"⏳ Loading {MODEL_ID} on {DEVICE} ...")
judge = CustomAutoJudge()
judge.from_pretrained(model_id=MODEL_ID, device=DEVICE, token=HF_TOKEN or None)
print(f"✅ Judge ready")

## 6 — Run Evaluation

For each category, each rubric is scored independently (1–5), then the category score is the **weighted mean** normalised to **0.0–1.0**.

In [ ]:
papers = {}

rubric_raw:       Dict[str, dict]  = {}  # individual rubric results
category_scores:  Dict[str, float] = {}  # aggregated per-category (0-1)

for cat_name, rubric_list in CATEGORIES.items():
    print(f"\n📂 Category: {cat_name.upper()}")
    cat_rubric_scores: List[Tuple[float, float]] = []

    for RubricClass, weight in rubric_list:
        rname = RubricClass.__name__
        if rname not in rubric_raw:
            print(f"   🔎 [{rname}] (w={weight:.3f}) ...")
            rubric = RubricClass(
                papers=papers,
                question=question,
                answer=report_md,
                domain=DOMAIN,
                vocabulary=VocabularyInjector(),
                example=ExampleInjector(),
            )
            raw = judge.judge(rubric=rubric, max_new_tokens=MAX_NEW_TOKENS)
            rating, rationale = parse_judge_result(raw, rname)
            rubric_raw[rname] = {'rating': rating, 'rationale': rationale, 'weight_in': {}}
            print(f"      rating={rating}/5")
        else:
            print(f"   ♻️  [{rname}] reusing cached result")
            rating = rubric_raw[rname]['rating']

        rubric_raw[rname]['weight_in'][cat_name] = weight
        cat_rubric_scores.append((float(rating), weight))

    cat_score = compute_category_score(cat_rubric_scores)
    category_scores[cat_name] = cat_score
    print(f"   ✅ {cat_name} score = {cat_score:.2f} (0-1)")

overall_score = round(sum(category_scores.values()) / len(category_scores), 4)
print(f"\n🏁 Overall score = {overall_score:.2f} (0-1)")

## 7 — Quality Dimensions Plot

6-panel bar chart that displays the scores for the rubric dimensions. All scores on **0–1 scale**.

In [ ]:
PANEL_TITLES = {
    'depth':      'Research Depth Score',
    'breadth':    'Research Breadth Score',
    'rigor':      'Scientific Rigor Score',
    'innovation': 'Innovation Score',
    'gap':        'Research Gap Score',
    'overall':    'Overall Quality Score',
}

ALL_CONFIGS = ['d1_b1', 'd1_b4', 'd4_b1', 'd4_b4']
xs = range(len(ALL_CONFIGS))

plot_scores = {k: category_scores.get(k, 0.0) for k in list(PANEL_TITLES.keys())[:-1]}
plot_scores['overall'] = overall_score

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, (cat, panel_title) in enumerate(PANEL_TITLES.items()):
    ax  = axes[i]
    val = plot_scores[cat]

    # Build bar values — current config gets the real score, others get 0
    bar_vals = [val if cfg == config_str else 0.0 for cfg in ALL_CONFIGS]

    ax.bar(xs, bar_vals, color='#2980b9', width=0.5)
    ax.set_title(panel_title, fontsize=12)
    ax.set_xticks(list(xs))
    ax.set_xticklabels(ALL_CONFIGS, fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score (0..1)', fontsize=9)

    # Label each bar
    for x, v in zip(xs, bar_vals):
        ax.text(x, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontsize=9)

engine = re.search(r'\\d+_(.*?)_orkg', stem)
engine_str = engine.group(1) if engine else stem
fig.suptitle(
    f'Research Quality Dimensions Analysis — {DOMAIN} / {engine_str} (n=1)',
    fontsize=15
)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
print("📊 Plot rendered")

## 8 — Final Evaluation Output (JSON)

In [ ]:
final_output = {
    'report':          str(report_path),
    'report_number':   report_num,
    'config':          config_str,
    'domain':          DOMAIN,
    'question':        question,
    'model':           MODEL_ID,
    'rubric_scores':   rubric_raw,
    'category_scores': category_scores,
    'overall':         overall_score,
}
display(Markdown('---\n### 🏁 Final Output\n---'))
print(json.dumps(final_output, indent=2, ensure_ascii=False))

## 9 — Save Outputs to Disk

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
# JSON
json_path = out_dir / f'{stem}_scores.json'
json_path.write_text(json.dumps(final_output, indent=2, ensure_ascii=False), encoding='utf-8')

# CSV
csv_path = out_dir / f'{stem}_scores.csv'
row = {'report': stem, 'config': config_str, 'title': title, 'question': question}
for cat, score in category_scores.items():
    row[f'{cat}_score'] = round(score, 4)
row['overall_score'] = round(overall_score, 4)
with csv_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)

# Plot
fig_path = out_dir / f'{stem}_quality_dimensions.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')

print(f"💾 JSON  → {json_path}")
print(f"💾 CSV   → {csv_path}")
print(f"💾 Plot  → {fig_path}")